# Mean F1 score per field

Reads an `evaluation_results.jsonl` (one JSON object per evaluated image) and reports the
F1 score per extraction field, broken down by document type
(`BANK_STATEMENT`, `INVOICE`, `RECEIPT`).

Columns in the summary tables:

* **mean_f1** — the plain mean of the per-document F1 scores (macro average). Every document
  counts once, regardless of how many line items it has.
* **std_f1** — sample standard deviation (ddof=1) of those per-document scores, i.e. how
  consistent the field is across documents. NaN when a group has only one document.
* **min_f1 / max_f1** — the observed range, which is more honest than sd when n is small and
  the distribution is skewed (a few total failures against a wall of 1.0s).
* **micro_f1** — computed from the pooled `tp`/`fp`/`fn` counts. Every *item* counts once, so
  documents with long transaction tables dominate.

For list-valued fields (transaction rows) macro and micro can differ a lot; quote which one
you mean.

In [ ]:
from pathlib import Path
import json

import pandas as pd

# Point this at the results file you want to analyse.
RESULTS_PATH = Path("/Users/tod/Desktop/evaluation_data/output/evaluation_results.jsonl")

In [ ]:
def load_field_scores(path: Path) -> pd.DataFrame:
    """Flatten an evaluation_results.jsonl into one row per (image, field).

    Args:
        path: Path to the JSONL results file.

    Returns:
        Long-format frame with image_name, document_type, field, f1_score, tp, fp, fn.
    """
    rows = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            for field, scores in record["field_scores"].items():
                rows.append(
                    {
                        "image_name": record["image_name"],
                        "document_type": record["document_type"],
                        "field": field,
                        "f1_score": scores["f1_score"],
                        "tp": scores["tp"],
                        "fp": scores["fp"],
                        "fn": scores["fn"],
                    }
                )
    if not rows:
        raise ValueError(f"No records with field_scores found in {path}")
    return pd.DataFrame(rows)


scores = load_field_scores(RESULTS_PATH)
print(f"{scores['image_name'].nunique()} documents, {scores['field'].nunique()} distinct fields")
print(scores.groupby("document_type")["image_name"].nunique().to_string())
scores.head()

In [ ]:
def summarise(frame: pd.DataFrame, *, by: list[str]) -> pd.DataFrame:
    """Aggregate F1 dispersion and micro F1 over the given grouping columns.

    Args:
        frame: Long-format frame from load_field_scores.
        by: Columns to group on, e.g. ["document_type", "field"].

    Returns:
        One row per group with n_docs, mean_f1, std_f1, min_f1, max_f1, micro_f1 and
        pooled counts. std_f1 is the sample standard deviation (ddof=1) of the
        per-document F1 scores, so it is NaN for any group holding a single document.
    """
    grouped = frame.groupby(by, dropna=False)
    summary = grouped.agg(
        n_docs=("image_name", "nunique"),
        mean_f1=("f1_score", "mean"),
        std_f1=("f1_score", "std"),
        min_f1=("f1_score", "min"),
        max_f1=("f1_score", "max"),
        tp=("tp", "sum"),
        fp=("fp", "sum"),
        fn=("fn", "sum"),
    )
    denominator = 2 * summary["tp"] + summary["fp"] + summary["fn"]
    summary["micro_f1"] = (2 * summary["tp"] / denominator).where(denominator > 0, 0.0)
    return summary[
        ["n_docs", "mean_f1", "std_f1", "min_f1", "max_f1", "micro_f1", "tp", "fp", "fn"]
    ]


per_type = summarise(scores, by=["document_type", "field"]).sort_values(
    ["document_type", "mean_f1"]
)
per_type.round(4)

In [ ]:
# Headline summary table: one row per field, mean F1 +/- standard deviation.
overall = summarise(scores, by=["field"]).sort_values("mean_f1")
summary_table = overall[["n_docs", "mean_f1", "std_f1", "min_f1", "max_f1"]].round(4)
summary_table.insert(
    1,
    "mean_pm_std",
    overall.apply(lambda r: f"{r.mean_f1:.3f} ± {r.std_f1:.3f}", axis=1),
)
summary_table

In [ ]:
# Mean F1 (and its spread) as a field x document-type matrix.
# NaN = field not evaluated for that document type.
matrix = per_type["mean_f1"].unstack("document_type")
errors = per_type["std_f1"].unstack("document_type")
matrix.round(4)

In [ ]:
# Error bars are +/- 1 standard deviation across documents, so they clip outside [0, 1].
ax = matrix.plot.barh(
    figsize=(9, 0.45 * len(matrix) + 2),
    xlim=(0, 1),
    xerr=errors.fillna(0.0),
    capsize=3,
    error_kw={"elinewidth": 1, "ecolor": "0.3"},
)
ax.set_xlabel("mean F1 (± 1 sd)")
ax.set_ylabel("")
ax.set_title(f"Mean F1 per field — {RESULTS_PATH.name}")
ax.legend(title="document type", loc="lower right")
ax.grid(axis="x", alpha=0.3)

In [ ]:
# Worst documents per field — the usual starting point for error analysis.
scores.sort_values("f1_score").head(15)[
    ["document_type", "field", "image_name", "f1_score", "tp", "fp", "fn"]
]